<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 05. MAE: 75%를 가리고 복원하기

## Masked Autoencoders Are Scalable Vision Learners

- **원 논문:** [Masked Autoencoders Are Scalable Vision Learners](https://arxiv.org/abs/2111.06377)
- **저자 / 발표:** Kaiming He, Xinlei Chen, Saining Xie, Yanghao Li, Piotr Dollár, Ross Girshick · CVPR (2022)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** ViT patch, masking, autoencoder
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** patchify, 75% random mask, visible-only encoder, lightweight decoder의 masked-pixel MSE를 구현합니다.

**축소하거나 생략한 부분:** ImageNet ViT-L/H와 800 epochs 대신 8×8 grayscale·16 patch·작은 attention decoder를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3 및 Figure 1 | image를 non-overlapping patch token으로 변환·복원 | patchify/unpatchify round-trip·patch shape |
| §3.1 및 Figure 1 | random patch masking 75% | visible/masked index |
| §3.2 | mask token 없는 visible-only encoder | encoder token 수 |
| §3.3 | mask token을 더한 lightweight decoder | 원래 patch 순서 |
| §3.4 | masked patch pixel MSE | loss·reconstruction |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_{MAE}=\frac{1}{|\mathcal M|}\sum_{i\in\mathcal M}\left\|x_i-\hat x_i\right\|_2^2$$

- **기호 정의:** M은 가려진 patch index, x_i와 x̂_i는 원본·복원 patch입니다.
- **수식의 역할:** encoder에는 visible patch만 넣고 작은 decoder가 masked patch를 복원합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3.1 및 Figure 1을 구현합니다.
- **입출력 shape:** image [B, 1, 8, 8] → patch [B, 16, 4] → 복원 [B, 16, 4]
- **평가:** masked-only MSE와 원본/복원 시각화
- **원문 대비 한계:** ImageNet ViT-L/H와 800 epochs 대신 8×8 grayscale·16 patch·작은 attention decoder를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3 및 Figure 1
- **이 셀의 책임:** image를 non-overlapping patch token으로 변환·복원
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** patchify/unpatchify round-trip·patch shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def patchify(x: Tensor, patch=2):
    return (
        x.unfold(2, patch, patch)
        .unfold(3, patch, patch)
        .permute(0, 2, 3, 1, 4, 5)
        .reshape(len(x), -1, patch * patch)
    )


def unpatchify(patches: Tensor, patch=2):
    side = int(patches.shape[1] ** 0.5)
    return (
        patches.reshape(len(patches), side, side, 1, patch, patch)
        .permute(0, 3, 1, 4, 2, 5)
        .reshape(len(patches), 1, side * patch, side * patch)
    )


p = patchify(images[:8])
assert p.shape == (8, 16, 4) and torch.allclose(unpatchify(p), images[:8])

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1 및 Figure 1
- **이 셀의 책임:** random patch masking 75%
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** visible/masked index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def random_mask(batch: int, n_patches: int, ratio=0.75):
    order = torch.stack([torch.randperm(n_patches) for _ in range(batch)])
    keep = int(n_patches * (1 - ratio))
    return order[:, :keep], order[:, keep:]


visible, masked = random_mask(8, 16)
assert visible.shape == (8, 4) and masked.shape == (8, 12)

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2 / §3.3
- **이 셀의 책임:** mask token 없는 visible-only encoder / mask token을 더한 lightweight decoder
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** encoder token 수 / 원래 patch 순서. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TinySelfAttention(nn.Module):
    def __init__(self, dim=16, heads=2):
        super().__init__()
        if dim % heads:
            raise ValueError("dim must be divisible by heads")
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)

    def forward(self, hidden):
        batch, tokens, dim = hidden.shape
        qkv = self.qkv(hidden).reshape(
            batch,
            tokens,
            3,
            self.heads,
            self.head_dim,
        )
        query, key, value = qkv.permute(2, 0, 3, 1, 4)
        scores = query @ key.transpose(-2, -1) / self.head_dim**0.5
        weights = scores.softmax(dim=-1)
        attended = weights @ value
        merged = attended.transpose(1, 2).reshape(batch, tokens, dim)
        return self.output(merged), weights


class TinyTransformerBlock(nn.Module):
    def __init__(self, dim=16, heads=2, ff_dim=32):
        super().__init__()
        self.attention_norm = nn.LayerNorm(dim)
        self.attention = TinySelfAttention(dim, heads)
        self.ff_norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, dim),
        )

    def forward(self, hidden):
        attended, weights = self.attention(self.attention_norm(hidden))
        hidden = hidden + attended
        return hidden + self.ff(self.ff_norm(hidden)), weights


class TinyMAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(4, 16)
        self.position = nn.Parameter(torch.randn(1, 16, 16) * 0.02)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, 16))
        self.decoder = TinyTransformerBlock(dim=16, heads=2, ff_dim=32)
        self.head = nn.Linear(16, 4)

    def forward(self, patches, visible, masked):
        visible = visible.to(patches.device)
        b = torch.arange(len(patches), device=patches.device)[:, None]
        visible_latent = (
            self.encoder(patches[b, visible]) + self.position[:, visible[0]]
            if torch.equal(visible, visible[:1].expand_as(visible))
            else self.encoder(patches[b, visible])
            + self.position.expand(len(patches), -1, -1)[b, visible]
        )
        full = self.mask_token.expand(len(patches), patches.shape[1], -1).clone()
        full[b, visible] = visible_latent
        decoded, _ = self.decoder(full + self.position)
        prediction = self.head(decoded)
        return prediction, visible_latent


mae = TinyMAE()
patches = patchify(images[:8])
pred, latent = mae(patches, visible, masked)
assert pred.shape == patches.shape and latent.shape[1] == 4

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.4
- **이 셀의 책임:** masked patch pixel MSE
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss·reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
mae = ACCELERATOR.move(mae)
optimizer = torch.optim.Adam(mae.parameters(), lr=0.008)
patches = ACCELERATOR.move(patchify(images[train_idx[:96]]))
losses = []
for _ in range(100):
    visible, masked = random_mask(len(patches), 16)
    visible, masked = ACCELERATOR.move(visible, masked)
    pred, _ = mae(patches, visible, masked)
    b = torch.arange(len(patches), device=DEVICE)[:, None]
    loss = F.mse_loss(pred[b, masked], patches[b, masked])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
visible, masked = random_mask(8, 16)
source = patchify(images[test_idx[:8]])
source_device, visible_device, masked_device = ACCELERATOR.move(source, visible, masked)
pred, _ = mae(source_device, visible_device, masked_device)
recon = source_device.clone()
b = torch.arange(8, device=DEVICE)[:, None]
recon[b, masked_device] = pred[b, masked_device].detach()
recon = recon.cpu()
assert losses[-1] < losses[0]
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    ax[0, i].imshow(images[test_idx[i], 0], cmap="gray")
    ax[1, i].imshow(unpatchify(recon)[i, 0], cmap="gray")
    ax[0, i].axis("off")
    ax[1, i].axis("off")
ax[0, 0].set_ylabel("input")
ax[1, 0].set_ylabel("MAE")
plt.tight_layout()
plt.show()
print(f"masked MSE {losses[0]:.4f} → {losses[-1]:.4f}")

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3.1 및 Figure 1 / §3.2 / §3.3 / §3.4
- **이 셀의 책임:** random patch masking 75% / mask token 없는 visible-only encoder / mask token을 더한 lightweight decoder / masked patch pixel MSE
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** visible/masked index / encoder token 수 / 원래 patch 순서 / loss·reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 20
    mask_ratio: float = 0.75
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(
    batch: Tensor,
    mask_ratio: float,
    generator: torch.Generator,
) -> tuple[Tensor, Tensor]:
    patches = patchify(batch)
    patch_count = patches.shape[1]
    masked_count = round(patch_count * mask_ratio)
    if not 0 < masked_count < patch_count:
        raise ValueError("mask_ratio must leave visible and masked patches")
    noise = torch.rand(
        len(batch),
        patch_count,
        generator=generator,
        device=batch.device,
    )
    mask = noise.argsort(dim=1)[:, :masked_count]
    assert mask.shape == (len(batch), masked_count)
    return patches, mask


class MaePortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.encoder = nn.Linear(4, config.hidden_dim)
        self.context = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.decoder = nn.Linear(config.hidden_dim, 4)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, config.hidden_dim))
        self.position = nn.Parameter(torch.randn(1, 16, config.hidden_dim) * 0.02)

    def forward(self, patches: Tensor, mask: Tensor) -> Tensor:
        visible = torch.ones(patches.shape[:2], dtype=torch.bool, device=patches.device)
        visible.scatter_(1, mask, False)
        visible_count = patches.shape[1] - mask.shape[1]
        visible_index = visible.float().argsort(dim=1, descending=True)[
            :, :visible_count
        ]
        patch_index = visible_index.unsqueeze(-1).expand(-1, -1, patches.shape[-1])
        visible_patches = patches.gather(1, patch_index)
        position_index = visible_index.unsqueeze(-1).expand(
            -1, -1, self.config.hidden_dim
        )
        visible_position = self.position.expand(len(patches), -1, -1).gather(
            1, position_index
        )
        visible_hidden = self.encoder(visible_patches) + visible_position

        # MAE §3.3: mask token을 masked 위치에 놓고, visible latent를
        # 원래 patch index에 scatter한 뒤 lightweight decoder로 복원합니다.
        decoder_tokens = self.mask_token.expand(len(patches), patches.shape[1], -1)
        decoder_tokens = decoder_tokens + self.position
        decoder_tokens = decoder_tokens.scatter(1, position_index, visible_hidden)
        visible_context = visible_hidden.mean(dim=1, keepdim=True)
        decoded = F.gelu(self.context(decoder_tokens + visible_context))
        return self.decoder(decoded)

    def compute_loss(self, prediction: Tensor, patches: Tensor, mask: Tensor) -> Tensor:
        batch = torch.arange(len(patches), device=patches.device)[:, None]
        return F.mse_loss(prediction[batch, mask], patches[batch, mask])

    def training_step(self, patches: Tensor, mask: Tensor) -> Tensor:
        prediction = self(patches, mask)
        return self.compute_loss(prediction, patches, mask)

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** §3.1 및 Figure 1 / §3.4
- **이 셀의 책임:** random patch masking 75% / masked patch pixel MSE
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** visible/masked index / loss·reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: MaePortfolioModel, batch: Tensor) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    generator = torch.Generator(device=batch.device).manual_seed(31)
    audit_generator = torch.Generator(device=batch.device).manual_seed(31)
    history = []
    previous_mask = None
    for _ in range(model.config.steps):
        patches, mask = prepare_views(
            batch,
            model.config.mask_ratio,
            generator,
        )
        _, repeated_mask = prepare_views(
            batch,
            model.config.mask_ratio,
            audit_generator,
        )
        expected_count = round(patches.shape[1] * model.config.mask_ratio)
        assert mask.shape[1] == expected_count
        assert torch.equal(mask, repeated_mask)
        if previous_mask is not None:
            assert not torch.equal(mask, previous_mask)
        previous_mask = mask.clone()
        loss = model.training_step(patches, mask)
        optimizer.zero_grad()
        loss.backward()
        assert model.mask_token.grad is not None
        assert float(model.mask_token.grad.abs().sum()) > 0.0
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** §3.1 및 Figure 1 / §3.4
- **이 셀의 책임:** random patch masking 75% / masked patch pixel MSE
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,P] → reconstruction [B,N,P]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** visible/masked index / loss·reconstruction. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: MaePortfolioModel, batch: Tensor
) -> dict[str, float]:
    with torch.no_grad():
        evaluation_generator = torch.Generator(
            device=batch.device,
        ).manual_seed(1031)
        patches, mask = prepare_views(
            batch,
            model.config.mask_ratio,
            evaluation_generator,
        )
        prediction = model(patches, mask)
        masked_mse = model.compute_loss(prediction, patches, mask)
        batch_index = torch.arange(len(patches), device=patches.device)[:, None]
        masked_prediction = prediction[batch_index, mask]
        position_variation = masked_prediction.std(dim=1).mean()
    return {
        "masked_patch_mse": float(masked_mse.cpu()),
        "masked_position_variation": float(position_variation.cpu()),
        "masked_patch_count": float(mask.shape[1]),
    }


portfolio_train_index = train_idx[:64]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_test_images = ACCELERATOR.move(
    images[portfolio_train_index],
    images[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(MaePortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(portfolio_model, portfolio_train_images)
portfolio_metrics = evaluate_portfolio_model(portfolio_model, portfolio_test_images)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["masked_position_variation"] > 0.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ImageNet ViT-L/H와 800 epochs 대신 8×8 grayscale·16 patch·작은 attention decoder를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.